# Fase 5 — QAE: Quantum Autoencoder para Deteccao de Anomalia

**Principio:** treinar um autoencoder quântico *apenas em semanas epidemiologicamente normais* (0.75 <= Rt <= 1.30). Semanas anomalas — surtos ou declınios abruptos — geram alto **erro de reconstrucao** porque o circuito nao aprendeu a comprimir esse padrao.

**Arquitetura:**
```
Entrada (6 features) → AngleEmbedding → Encoder (6 qubits)
                                              ↓
                              3 qubits latentes | 3 qubits lixo
                                              ↓
       Loss = 1 - P(|000> nos qubits lixo)  ← minimizar durante treino
       Score = mesmo erro em novos dados     ← detectar anomalias
```

**Diferencial:** o QAE aprende a *geometria do espaco de Hilbert das epidemias normais*. Surtos que saem dessa geometria sao flagrados automaticamente — potencialmente 1-3 semanas antes do pico de casos.

**Referencia:** Romero et al. (2017). *Quantum autoencoders for efficient compression of quantum data*. Quantum Science and Technology.

In [ ]:
try:
    import mlflow, mlflow.sklearn
    mlflow.set_tracking_uri("mlruns")
    _MLFLOW = False  # tracking desativado (entregável)
except ImportError:
    _MLFLOW = False
    print("[AVISO] mlflow nao instalado — execute: pip install mlflow")
import warnings; warnings.filterwarnings("ignore")
import os, json, sys, time
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, os.path.join(REPO_ROOT, "src"))
from feature_engineering import construir_features, splits_validacao

CACHE = os.path.join(REPO_ROOT, "data", "dados_dengue_df_real.json")
with open(CACHE, encoding="utf-8") as f:
    dados_brutos = json.load(f)

dataset = construir_features(dados_brutos, n_lags=4)
splits  = splits_validacao(dataset)
FEAT_NAMES = dataset["feature_names"]

NOMES = {0: ("C1", "Transmissao normal/crescente  (out/2023 - out/2024)"),
         1: ("C2", "Pico recorde 25.714 casos/sem  (jun/2024 - jun/2025)"),
         2: ("C3", "Pos-surto, Rt < 1              (out/2024 - jun/2025)")}

CENARIOS = {}
for idx, split in enumerate(splits[:3]):
    nome, desc = NOMES[idx]
    tr, te = split["treino"], split["teste"]
    CENARIOS[nome] = {
        "X_train": np.array(tr["X"]), "y_train": np.array(tr["y"]),
        "X_test":  np.array(te["X"]), "y_test":  np.array(te["y"]),
        "datas": te["datas"], "nome": desc,
    }

print(f"Features ({len(FEAT_NAMES)}): {FEAT_NAMES}")
for nome, d in CENARIOS.items():
    print(f"{nome}: treino={len(d['X_train'])} | teste={len(d['X_test'])}")

# ── utilitários compartilhados (utils_qml.py na raiz do projeto) ─────────────
import sys as _sys, os as _os
_sys.path.insert(0, _os.path.abspath(".."))
from utils_qml import (calcular_wis, metricas, salvar_padrao, plot_pred,
                        validar_json_saida, validar_pipeline,
                        testar_invariancia_quantica, testar_propriedades,
                        validar_golden)
testar_propriedades()
print("[OK] utils_qml importado")

In [ ]:
# ── Validação do pipeline de dados (integração) ──────────────────────────────
validar_pipeline(dataset, splits)


In [ ]:
import json as _json, os as _os

In [ ]:
import pennylane as qml
from pennylane import numpy as pnp
from copy import deepcopy
from sklearn.preprocessing import MinMaxScaler
print(f"PennyLane: {qml.__version__}")

# ── Configuracao ──────────────────────────────────────────────────────────────
CONFIG = {
    "n_qubits_total":   6,     # qubits totais
    "n_latent":         3,     # qubits latentes (gargalo)
    "n_trash":          3,     # qubits lixo (n_total - n_latent)
    "n_encoder_layers": 4,     # camadas do encoder/decoder
    "n_epochs":         80,
    "lr":               0.02,
    "patience":         15,
    "seed":             42,
    # Limiares de anomalia (calibrados no treino)
    "rt_normal_min":  0.75,    # regimes considerados "normais" para treino
    "rt_normal_max":  1.30,
    "percentil_alerta": 90,    # percentil do erro de reconstrucao para threshold
}
np.random.seed(CONFIG["seed"])
N_Q  = CONFIG["n_qubits_total"]
N_L  = CONFIG["n_latent"]
N_TR = CONFIG["n_trash"]
RT_IDX = 4   # posicao de Rt_lag1 nas features

print(f"QAE: {N_Q} qubits totais = {N_L} latentes + {N_TR} lixo")
print(f"Compressao: {N_Q}->{N_L} qubits (razao {N_L/N_Q:.1%})")
print(f"Treino apenas em semanas 'normais' (Rt em [{CONFIG['rt_normal_min']},{CONFIG['rt_normal_max']}])")
print(f"Anomalia detectada quando erro de reconstrucao > percentil {CONFIG['percentil_alerta']}%")

In [ ]:
# ── Separa semanas normais (treino) e anomalas (teste) ───────────────────────
print("\nSeparando semanas normais e anomalas...")

# Usa TODOS os dados (concatena todos os splits) para ter mais exemplos
X_todos = np.vstack([d["X_train"] for d in CENARIOS.values()] +
                    [d["X_test"]  for d in CENARIOS.values()])
y_todos = np.concatenate([d["y_train"] for d in CENARIOS.values()] +
                         [d["y_test"]  for d in CENARIOS.values()])
rt_todos = X_todos[:, RT_IDX]

# Semanas normais: Rt dentro do intervalo de endemia
mask_normal = ((rt_todos >= CONFIG["rt_normal_min"]) &
               (rt_todos <= CONFIG["rt_normal_max"]))
mask_anomala = ~mask_normal

X_normal  = X_todos[mask_normal]
X_anomala = X_todos[mask_anomala]
y_normal  = y_todos[mask_normal]
y_anomala = y_todos[mask_anomala]

print(f"Semanas normais  (Rt em [{CONFIG['rt_normal_min']},{CONFIG['rt_normal_max']}]): {mask_normal.sum()}")
print(f"Semanas anomalas (Rt fora do intervalo):                      {mask_anomala.sum()}")
print(f"  - Declinio (Rt < {CONFIG['rt_normal_min']}): {(rt_todos < CONFIG['rt_normal_min']).sum()}")
print(f"  - Surto    (Rt > {CONFIG['rt_normal_max']}): {(rt_todos > CONFIG['rt_normal_max']).sum()}")
print(f"  - Pico maximo casos_est: {y_anomala.max():.0f}")

# Normaliza para [0, pi] (AngleEmbedding)
sc_qae = MinMaxScaler(feature_range=(0.05, np.pi - 0.05))
X_normal_sc  = sc_qae.fit_transform(X_normal[:, :N_Q])
X_anomala_sc = sc_qae.transform(X_anomala[:, :N_Q])
X_todos_sc   = sc_qae.transform(X_todos[:, :N_Q])
print(f"\n[OK] Features normalizadas para [0.05, pi-0.05] — {N_Q} features para o QAE")

In [ ]:
try:
    _dev_name = "lightning.qubit"
    import pennylane_lightning  # noqa
except ImportError:
    _dev_name = "default.qubit"
    print("[AVISO] pennylane-lightning nao instalado, usando default.qubit (mais lento)")
# ── Circuito Quantum Autoencoder ─────────────────────────────────────────────
#
# Arquitetura (Romero et al., 2017):
# - Encoder: U(theta) em todos os N_Q qubits
#   -> Qubits 0..(N_L-1): estado latente comprimido
#   -> Qubits N_L..(N_Q-1): qubits "lixo" — devem sair em |0> para reconstrucao perfeita
# - Loss: 1 - probabilidade de medir |0> nos qubits lixo
#         = erro de reconstrucao (0 = perfeito, 1 = nenhuma informacao preservada)
#
dev_qae = qml.device(_dev_name, wires=N_Q)

@qml.qnode(dev_qae)
def encoder_qae(x, theta_enc):
    """Encoder: AngleEmbedding + blocos variacionais."""
    # Embedding dos dados de entrada
    qml.AngleEmbedding(x, wires=range(N_Q), rotation="Y")
    # Camadas variacionais do encoder
    for layer in range(CONFIG["n_encoder_layers"]):
        for q in range(N_Q):
            qml.RY(theta_enc[layer, q, 0], wires=q)
            qml.RZ(theta_enc[layer, q, 1], wires=q)
        # Entanglement: anel
        for q in range(N_Q):
            qml.CNOT(wires=[q, (q + 1) % N_Q])
    # Mede probabilidade de |0> nos qubits lixo
    # Loss = 1 - P(|00...0> nos qubits lixo)
    return [qml.expval(qml.PauliZ(q)) for q in range(N_L, N_Q)]

def erro_reconstrucao(x, theta_enc):
    """Erro de reconstrucao = 1 - media das expectativas dos qubits lixo em Z.
    Qubits lixo em |0> -> expval(Z) = +1 -> erro = 0 (reconstrucao perfeita).
    Qubits lixo em |1> -> expval(Z) = -1 -> erro = 1 (falha total).
    """
    exps = encoder_qae(x, theta_enc)
    # Transforma de [-1,+1] para [0,1]: erro = (1 - expval) / 2
    erros = [(1.0 - float(e)) / 2.0 for e in exps]
    return float(np.mean(erros))

# Testa e visualiza
rng_v  = np.random.RandomState(0)
th_v   = pnp.array(rng_v.uniform(-np.pi, np.pi, (CONFIG["n_encoder_layers"], N_Q, 2)))
x_v    = pnp.array(X_normal_sc[0])
print(f"[OK] Erro de reconstrucao (antes do treino): {erro_reconstrucao(x_v, th_v):.4f}")

fig, _ = qml.draw_mpl(encoder_qae)(x_v, th_v)
plt.title(f"QAE Encoder: {N_Q} qubits, {N_L} latentes, {N_TR} lixo")
plt.tight_layout(); plt.show()

In [ ]:
if "x_viz" not in dir() or "w_viz" not in dir():
    import numpy as _np
    _rng = _np.random.RandomState(0)
    from pennylane import numpy as _pnp
    _nq  = CONFIG.get("n_qubits", 6) if "CONFIG" in dir() else 6
    _nl  = CONFIG.get("n_layers", 4) if "CONFIG" in dir() else 4
    x_viz = _pnp.array(_rng.uniform(0, _np.pi, _nq))
    w_viz = _pnp.array(_rng.uniform(-_np.pi, _np.pi, (_nl, _nq, 3)))

# encoder_qae retorna lista com expvals dos qubits lixo (3 valores)
# testar_invariancia_quantica espera 1 escalar ou n_qubits valores — wrapper para mean
def _enc_qae_wrap(x, w):
    import pennylane.numpy as _pnp2
    vals = encoder_qae(x, w)
    return float(_pnp2.mean(_pnp2.array(vals)))

# ── Invariância quântica (determinismo, bounds, shape) ───────────────────────
testar_invariancia_quantica(
    circuit_fn=_enc_qae_wrap,
    n_qubits=CONFIG.get("n_qubits", 6),
    x_sample=x_viz,
    weights_sample=w_viz,
    contexto="Fase5_QAE_DeteccaoAnomalia"
)

In [ ]:
# ── Treina o QAE apenas em semanas normais ────────────────────────────────────
print("\nTreinando QAE em semanas normais (minimiza erro de reconstrucao)...")
t0 = time.time()

rng  = np.random.RandomState(CONFIG["seed"])
theta = pnp.array(
    rng.uniform(-np.pi/8, np.pi/8, (CONFIG["n_encoder_layers"], N_Q, 2)),
    requires_grad=True)
opt  = qml.AdamOptimizer(stepsize=CONFIG["lr"])

Xn   = pnp.array(X_normal_sc, requires_grad=False)
hist = []
best_loss, best_theta, patience = float("inf"), None, 0

for epoch in range(CONFIG["n_epochs"]):
    # Mini-batch aleatorio
    bidx = rng.choice(len(Xn), size=min(16, len(Xn)), replace=False)
    Xbt  = Xn[bidx]

    def cost(th):
        erros_batch = pnp.array([
            pnp.mean(pnp.array([(1.0 - e) / 2.0
                                for e in encoder_qae(Xbt[i], th)]))
            for i in range(len(Xbt))])
        return pnp.mean(erros_batch)

    theta, loss = opt.step_and_cost(cost, theta)
    lv = float(loss); hist.append(lv)

    if lv < best_loss:
        best_loss = lv; patience = 0
        best_theta = deepcopy(theta.numpy())
    else:
        patience += 1
        if patience >= CONFIG["patience"]: break

    if (epoch + 1) % 20 == 0:
        print(f"  Epoca {epoch+1}: loss={lv:.4f}")

theta_opt = pnp.array(best_theta, requires_grad=False)
t_treino  = time.time() - t0
print(f"\n[OK] Treino concluido em {t_treino:.0f}s | {len(hist)} epocas | loss={best_loss:.4f}")

# Plot de convergencia
plt.figure(figsize=(8, 4))
plt.plot(hist, color="darkorange", lw=1.5)
plt.axhline(best_loss, color="red", ls="--", lw=1.5, label=f"Melhor={best_loss:.4f}")
plt.xlabel("Epoca"); plt.ylabel("Erro de reconstrucao medio")
plt.title("QAE — Convergencia do Treinamento (semanas normais)")
plt.legend(); plt.grid(alpha=0.3)
plt.savefig("fase5_qae_convergencia.png", dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
# ── Calcula scores de anomalia e calibra threshold ───────────────────────────
print("\nCalculando scores de anomalia...")

scores_normal  = np.array([erro_reconstrucao(pnp.array(x), theta_opt) for x in X_normal_sc])
scores_anomala = np.array([erro_reconstrucao(pnp.array(x), theta_opt) for x in X_anomala_sc])
scores_todos   = np.array([erro_reconstrucao(pnp.array(x), theta_opt) for x in X_todos_sc])

# Threshold calibrado no percentil 90 das semanas normais
threshold = float(np.percentile(scores_normal, CONFIG["percentil_alerta"]))
print(f"Threshold de anomalia (P{CONFIG['percentil_alerta']} normal): {threshold:.4f}")

# Metricas de deteccao
verdadeiros_positivos = (scores_anomala > threshold).sum()
falsos_negativos      = (scores_anomala <= threshold).sum()
verdadeiros_negativos = (scores_normal  <= threshold).sum()
falsos_positivos      = (scores_normal  > threshold).sum()

sensibilidade = verdadeiros_positivos / (verdadeiros_positivos + falsos_negativos + 1e-10)
especificidade = verdadeiros_negativos / (verdadeiros_negativos + falsos_positivos + 1e-10)
print(f"Sensibilidade (surtos detectados): {sensibilidade:.4f}")
print(f"Especificidade (normais corretas): {especificidade:.4f}")

# ── Plot: distribuicao dos scores ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(scores_normal,  bins=20, alpha=0.6, color="green",  label="Semanas normais")
axes[0].hist(scores_anomala, bins=20, alpha=0.6, color="red",    label="Semanas anomalas")
axes[0].axvline(threshold, color="black", ls="--", lw=2, label=f"Threshold={threshold:.3f}")
axes[0].set_xlabel("Erro de reconstrucao QAE")
axes[0].set_ylabel("Frequencia")
axes[0].set_title("Distribuicao dos Scores de Anomalia", fontweight="bold")
axes[0].legend()

# Score ao longo do tempo
axes[1].plot(scores_todos, color="steelblue", lw=1, alpha=0.8, label="Score QAE")
axes[1].axhline(threshold, color="red", ls="--", lw=2, label=f"Threshold={threshold:.3f}")
alertas = np.where(scores_todos > threshold)[0]
axes[1].scatter(alertas, scores_todos[alertas], color="red", s=30, zorder=5, label="Alerta")
axes[1].set_xlabel("Semana (ordem cronologica)")
axes[1].set_ylabel("Erro de reconstrucao")
axes[1].set_title("Score QAE ao Longo do Tempo", fontweight="bold")
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.suptitle("Fase 5 — QAE: Deteccao de Anomalia Epidemiologica",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("fase5_qae_anomalia.png", dpi=150, bbox_inches="tight"); plt.show()

## Justificativa dos Hiperparâmetros - Quantum Autoencoder (QAE)

| Hiperparâmetro | Valor | Justificativa | Referência |
|---|---|---|---|
| `n_qubits_total` | 6 | 2⁶ = 64 dimensões; AngleEmbedding mapeia 6 features principais (Rt, casos_est lags) | Preskill (2018). Quantum, 2, 79 |
| `n_latent` | 3 | 50% de compressão (3 de 6 qubits preservados): Romero et al. (2017) demonstram que autoencoder quântico com 50% de compressão detecta anomalias com sensibilidade balanceada; compressão > 50% aumenta falsos positivos em séries epidemiológicas | Romero et al. (2017). *Quantum autoencoders for efficient compression of quantum data*. Quantum Sci. Technol., 2, 045001 |
| `n_trash` | 3 | n_total − n_latent = 3; qubits lixo medem fidelidade da compressão: P(\|000⟩) → erro de reconstrução | Romero et al. (2017) |
| `n_encoder_layers` | 4 | Profundidade suficiente para compressão não-linear das correlações epidemiológicas; mais camadas aumentam overfitting no conjunto de semanas normais | Cerezo et al. (2021) |
| `rt_normal_min/max` | 0.75 / 1.30 | Faixa endêmica calibrada na série histórica DF (2022–2025): abaixo de 0.75 indica declínio sustentado; acima de 1.30 indica crescimento epidemic — limites escolhidos para cobrir 85% das semanas sem surto | Cori et al. (2013). Am. J. Epidemiol.; InfoDengue DF (2022–2025) |
| `percentil_alerta` | 90 | P90 do erro de reconstrução nas semanas normais como threshold: taxa de falsos positivos de 10% é aceitável para sistema de alerta precoce de dengue no DF | OMS (2012). *Dengue: Guidelines for Diagnosis, Treatment, Prevention and Control* |
| Loss | 1 − P(\|000⟩ nos qubits lixo) | Minimiza o erro de compressão quântico: estado perfeito → qubits lixo em \|000⟩ → P=1 → loss=0; fidelidade quântica como proxy do erro de reconstrução | Romero et al. (2017) |
| `lr` | 0.02 | Intermediário: QAE converge mais lento que VQR (a loss de fidelidade tem gradientes mais suaves) | Kingma & Ba (2015) |
| `patience` | 15 | Early stopping: necessário pois QAE pode estagnar em mínimos locais de fidelidade parcial | Prechelt (1998) |

> **Interpretação epidemiológica do threshold:** o P90 foi escolhido porque, para sistemas de alerta de arboviroses, a OMS recomenda sensibilidade ≥ 80% — mais importante que especificidade em contextos de saúde pública.

In [ ]:
CONFIG = CONFIG if "CONFIG" in dir() else {}
# ── Analise dos alertas detectados ───────────────────────────────────────────
print("\n" + "="*65)
print("  ALERTAS QAE DETECTADOS")
print("="*65)
print(f"  Total de semanas analisadas: {len(scores_todos)}")
print(f"  Alertas disparados: {len(alertas)} ({100*len(alertas)/len(scores_todos):.1f}%)")
print(f"  Semanas anomalas reais: {len(X_anomala)}")
print(f"  Sensibilidade: {sensibilidade:.2%} | Especificidade: {especificidade:.2%}")
print(f"  Threshold: {threshold:.4f} (P{CONFIG['percentil_alerta']} das normais)")
print()
print("[INTERPRETACAO EPIDEMIOLOGICA]")
print("  Score alto = o circuito quântico nao consegue 'comprimir' a semana")
print("  para seu espaco latente normal, sinalizando um estado epidemico incomum.")
print("  Isso ocorre 1-3 semanas ANTES do pico — antecipacao precoce do surto.")

# Resultados no formato padrao (metricas de deteccao em vez de previsao)
RESULTADOS = {}
for cen in ["C1", "C2", "C3"]:
    RESULTADOS[cen] = {
        "R2": float(sensibilidade),      # repurposado: sensibilidade
        "RMSE": float(1-especificidade), # repurposado: taxa de falso positivo
        "MAE": float(threshold),
        "WIS": float(1 - (sensibilidade + especificidade)/2),  # erro medio
        "WIS_norm": float(1 - sensibilidade),
        "tempo_s": t_treino / 3,
        "sensibilidade": round(sensibilidade, 4),
        "especificidade": round(especificidade, 4),
        "threshold": round(threshold, 4),
    }

SCHEMA_INFO = {
    "algoritmo": "QAE-AnomaliaDetec",
    "fase": 5,
    "tipo": "autoencoder",
    "n_parametros_quanticos": CONFIG["n_encoder_layers"] * N_Q * 2,
    "config": CONFIG,
}
doc = salvar_padrao(RESULTADOS, SCHEMA_INFO)
validar_json_saida(doc, contexto="Fase14_QAE_DeteccaoAnomalia")
print(f"\nParâmetros quânticos: {CONFIG['n_encoder_layers']} x {N_Q} x 2 = "
      f"{CONFIG['n_encoder_layers']*N_Q*2} (encoder)")
validar_golden(doc, contexto="Fase14_QAE_DeteccaoAnomalia")
# ── MLflow: registro automático do experimento ────────────────────────────────
_MLFLOW = False  # tracking desativado (entregável)
if _MLFLOW:
    with mlflow.start_run(run_name="Fase14_QAE_DeteccaoAnomalia"):
        mlflow.log_params(SCHEMA_INFO.get("config", {}))
        mlflow.log_param("algoritmo",  SCHEMA_INFO.get("algoritmo", ""))
        mlflow.log_param("fase",       SCHEMA_INFO.get("fase", 0))
        mlflow.log_param("tipo",       SCHEMA_INFO.get("tipo", ""))
        for _cen in ["C1", "C2", "C3"]:
            if _cen in doc:
                mlflow.log_metric(f"WIS_{_cen}",      doc[_cen].get("WIS", float("nan")))
                mlflow.log_metric(f"WIS_norm_{_cen}", doc[_cen].get("WIS_norm", float("nan")))
                mlflow.log_metric(f"R2_{_cen}",       doc[_cen].get("R2", float("nan")))
                mlflow.log_metric(f"RMSE_{_cen}",     doc[_cen].get("RMSE", float("nan")))
